# Imports

In [ ]:
from IPython.display import Image, display, SVG, HTML
from typing import Dict, List, Tuple, Generator, Optional, Any, Callable, Literal

from scipy.signal import medfilt, savgol_filter
from sklearn.cluster import KMeans

import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
import numpy as np
import os
import re
import time
import math
import datetime

from tqdm import tqdm
import hatchet as ht
import thicket as tt
from collections import defaultdict

In [ ]:
# Track how long the notebook takes to run
NOTEBOOK_START_TIME = time.time()
print(f"Notebook started at {NOTEBOOK_START_TIME} ({datetime.datetime.fromtimestamp(NOTEBOOK_START_TIME)})")

# Preamble

### Define The Name of the Cluster & Trace Paths

Define the name of the cluster and the paths to the trace files.
This is needed to parse the trace directory paths correctly as generated by the master run script.

In [ ]:
CLUSTER_NAME = "frontier"

STEP_VECTOR_SIZE = 134217728 # 2^27

ROOT_TRACE_DIR = "../frontier-results-2"

### Define Which Ranks Run on Each Node

Define the topology of the runs. This is needed to attribute energy consumption correctly. Metrics are gathered at the node level, so we
need to know which node each rank ran on to get the metrics associated with that node + rank.

In [ ]:
ACTIVE_PERIODS_MS = [1, 2, 5, 20, 50, 100, 200, 400, 1000]
IDLE_PERIODS_MS = ACTIVE_PERIODS_MS

DOUBLE_GCD_NODE_RANKS = {'node0': ['MPI Rank 0', 'MPI Rank 1', 'MPI Rank 2', 'MPI Rank 3', 'MPI Rank 4', 'MPI Rank 5', 'MPI Rank 6', 'MPI Rank 7']}
SINGLE_GCD_NODE_RANKS = {'node0': ['MPI Rank 0', 'MPI Rank 1', 'MPI Rank 2', 'MPI Rank 3']}

### Define The Metrics to Attribute to Each Rank

This defines which MPI rank is on which GCD (frontier) or APU (el capitan). Two ranks drawing from the same GPU (each GPU has 2 GCDs) will have their energy consumption estimated independently based on their exclusive runtime.

In [ ]:
DOUBLE_GCD_RANKS_TO_METRICS = {
	'MPI Rank 0': ['A2rocm_smi:::energy_count:device=0'],
	'MPI Rank 1': ['A2rocm_smi:::energy_count:device=0'],
	'MPI Rank 2': ['A2rocm_smi:::energy_count:device=2'],
	'MPI Rank 3': ['A2rocm_smi:::energy_count:device=2'],
	'MPI Rank 4': ['A2rocm_smi:::energy_count:device=4'],
	'MPI Rank 5': ['A2rocm_smi:::energy_count:device=4'],
	'MPI Rank 6': ['A2rocm_smi:::energy_count:device=6'],
	'MPI Rank 7': ['A2rocm_smi:::energy_count:device=6']
}

SINGLE_GCD_RANKS_TO_METRICS = {
    'MPI Rank 0': ['A2rocm_smi:::energy_count:device=0'],
    'MPI Rank 1': ['A2rocm_smi:::energy_count:device=2'],
    'MPI Rank 2': ['A2rocm_smi:::energy_count:device=4'],
    'MPI Rank 3': ['A2rocm_smi:::energy_count:device=6']
}

### Define Which Metrics to Track in the Traces

This defines the metrics we're going to pay attention to when parsing the traces.
All metrics that are used above must be included here.

In [ ]:

METRICS_TO_ATTRIBUTE = [
    'A2rocm_smi:::energy_count:device=0',
    'A2rocm_smi:::energy_count:device=2',
    'A2rocm_smi:::energy_count:device=4',
    'A2rocm_smi:::energy_count:device=6',
    
    'A2coretemp:::craypm:accel0_energy',
    'A2coretemp:::craypm:accel1_energy',
    'A2coretemp:::craypm:accel2_energy',
    'A2coretemp:::craypm:accel3_energy',
    
    'A2coretemp:::craypm:accel0_power',
    'A2coretemp:::craypm:accel1_power',
    'A2coretemp:::craypm:accel2_power',
    'A2coretemp:::craypm:accel3_power'
]

In [ ]:
def calculate_steps_from_periods(time_active, time_sleep, time_desired=15000):
    # local time_active=$1
    # local time_sleep=$2
    # local nsteps=$((15000 / (time_active + time_sleep))) # Target ~15s total time
    return time_desired // (time_active + time_sleep)  # Target ~15s total time

In [ ]:
def steps_for_active_and_idle(active_ms: int, idle_ms: int) -> int:
    x = math.floor(15001 / (active_ms + idle_ms))
    print(f"Calculating steps for active_ms={active_ms}, idle_ms={idle_ms}: {x}")
    return x


def get_trace_paths_for_step_variant(ranks: int) -> Dict[Tuple[int, int], List[str]]:
    result = {}
    # Instead, just iterate through all the traces and add them based on their time active and idle
    for d in os.listdir(f"{ROOT_TRACE_DIR}/"):
        print(f"Checking directory: {d}")
        # Get the time_active and time_idle from the directory name
        match = re.match(rf"step_function-r{ranks}---vector_size_{STEP_VECTOR_SIZE}_--n_steps_(\d+)_--time_active_(\d+)_--time_sleep_(\d+)", d)
        if match:
            n_steps = int(match.group(1))
            time_active = int(match.group(2))
            time_idle = int(match.group(3))
            
            trace_dir = f"{ROOT_TRACE_DIR}/{d}"
            for subdir in os.listdir(trace_dir):
                if re.match(rf"trial_\d+-{CLUSTER_NAME}\d+-.*-step_function-.*", subdir):
                    trace_path = f"{trace_dir}/{subdir}/traces.otf2"
                    if os.path.exists(trace_path):
                        key = (time_active, time_idle)
                        if key not in result:
                            result[key] = []
                        result[key].append(trace_path)
    return result

DOUBLE_GCD_TRACES = get_trace_paths_for_step_variant(ranks=8)
SINGLE_GCD_TRACES = get_trace_paths_for_step_variant(ranks=4)

print("DOUBLE_GCD_TRACES:", len(DOUBLE_GCD_TRACES))
print("SINGLE_GCD_TRACES:", len(SINGLE_GCD_TRACES))

### Metrics
Which metrics to trace, and functions for reading/writing them to CSVs.

In [ ]:
# def callgraph_to_csv(call_graph: CallGraph, group: str, thread: str, filename: str):
#     """Convert a CallGraph to a CSV file."""
#     with open(filename, 'w') as f:
#         f.write("Thread,Group,Depth,Name,Start Time,End Time,Duration\n")
#         intervals = call_graph.get_intervals_between(float('-inf'), float('inf'))
        
#         for interval in intervals:
#             start = interval.start
#             end = interval.end or float('inf')
#             duration = end - start
#             name = interval.name if interval.name else "Unknown"
#             depth = interval.depth if interval.depth else 0

#             f.write(f"{thread},{group},{depth},\"{name}\",{start},{end},{duration}\n")

def metrics_to_csv(group: str, thread_metrics: Dict[str, List[Tuple[float, float]]], filename: str):
    """Convert metrics to a CSV file."""
    with open(filename, 'w') as f:
        f.write("Group,Metric Name,Time,Value\n")
        for metric_name, values in thread_metrics.items():
            for time, value in values:
                f.write(f"{group},{metric_name},{time},{value}\n")

def read_metrics_dataframe(file: str) -> pd.DataFrame:
    df = pd.read_csv(file)
    # For all the rocm metrics, divide by 1e6 to convert from uJ to J
    for metric_name in df['Metric Name'].unique():
        if 'rocm_smi:::energy_count:device' in metric_name:
            # Divide by a million to convert from uJ to J
            # df.loc[df['Metric Name'] == metric_name, 'Value'] = np.divide(df.loc[df['Metric Name'] == metric_name, 'Value'], 1_000_000.0)
            mask = df['Metric Name'] == metric_name
            df['Value'] = df['Value'].astype(float)
            df.loc[mask, 'Value'] = (df.loc[mask, 'Value'] / 1_000_000.0).astype(float)
    return df

def read_call_graph_dataframe(file: str) -> pd.DataFrame:
    return pd.read_csv(file)


### Remove Conflicting CSVs
Remove any existing CSVs that might conflict with new ones to be generated.

In [ ]:
# Remove all the old CSV files
for file in os.listdir('.'):
    if file.endswith('.csv'):
        os.remove(file)

### Setup for Loading Traces
Convert an OTF2 to a CSV dataset to be loaded into a dataframe

In [ ]:
def trace_to_csv(trace: str, processes_to_track: List[str], metrics_to_track: List[str] = METRICS_TO_ATTRIBUTE, cray_time_offset=0.0):
	# Remove all the old CSV files
	for file in os.listdir('.'):
		if file.endswith('.csv'):
			os.remove(file)
	# Shell out to ./trace_to_csv_parallel binary with `./trace_to_csv_parallel --tracePath=/path/to/trace.otf2`
	# Shell out to ./trace_to_csv_parallel binary with `./trace_to_csv_parallel --tracePath=/path/to/trace.otf2`
	cmd = f'./trace_to_csv_parallel --trace={trace} --metrics="{",".join(metrics_to_track)}" > /dev/null 2>&1'
	if os.system(cmd):
		print(f'Error processing trace: {trace}')
		return

# Visualize the dataframes
def visualize_dataframe(df):
	"""Visualize the first few rows of a DataFrame as HTML."""
	# pd.set_option("display.width", 2400)
	# pd.set_option("display.max_colwidth", 100)
	# pd.set_option("display.max_rows", 20)
	
	display(HTML(df.head(n=20).to_html(index=False)))


### Math Helpers

Helpers for linearly interpolating values and computing numerical integrals + derivatives

In [ ]:
def sampled_to_continuous(df, time_col='Time', value_col='Value'):
    t_samples = df[time_col].astype(float)
    y = df[value_col].astype(float)
    # Apply a savgol filter to smooth the data
    # y = scipy.signal.savgol_filter(y, window_length=11, polyorder=2, mode='nearest')
    # Apply a median filter to smooth the data
    # y = scipy.signal.medfilt(y, kernel_size=11)
 
    # Create a continuous function that interpolates the sampled data
    f = lambda t: np.interp(t, t_samples, y)
    return f

def numerical_derivative(func, h=1e-5):
    """Compute the numerical derivative of a function at a point t."""
    return lambda t: (func(t + h) - func(t - h)) / (2 * h)

def numerical_integral(f, a=0.0, dx=1e-1):
    """
    Return a function F(x) that approximates the integral of f(t) dt from t=a to t=x,
    using the trapezoidal rule in a fully vectorized manner.
    The function F(x) will return a scalar when x is a scalar, and an array
    when x is an array.
    """
    # Determinar si f soporta arrays de NumPy
    try:
        _ = f(np.array([a, a + dx]))
        f_np = f
    except Exception:
        f_np = np.vectorize(f)
    
    def F(x):
        x_arr = np.atleast_1d(x).astype(float)
        # Manejo de x < a invirtiendo signo
        sign = np.ones_like(x_arr)
        mask = x_arr < a
        sign[mask] = -1
        x_pos = np.where(mask, a + (a - x_arr), x_arr)
        
        # Grid hasta el máximo de x
        xmax = x_pos.max()
        t = np.arange(a, xmax + dx, dx)
        u = f_np(t)
        
        # Suma acumulada de trapecios
        trap_heights = (u[:-1] + u[1:]) / 2
        cum = np.concatenate(([0], np.cumsum(trap_heights * dx)))
        
        # Índices y remanentes
        idx = ((x_pos - a) // dx).astype(int)
        rem = x_pos - (a + idx * dx)
        
        # Valores en fronteras
        f_end = f_np(x_pos)
        f_start = u[idx]
        
        # Integral aproximada
        integral = cum[idx] + (f_start + f_end) * rem / 2
        result = sign * integral
        
        # Devolver escalares cuando corresponda
        return result.item() if np.isscalar(x) else result
    
    return F
        
def continuous_savgol_filter(f, window_length=11, polyorder=2, sample_rate=0.001):
    """
    Return a smoothed version of the continuous function f using a Savitzky-Golay filter.
    The function f is sampled at intervals of sample_rate to apply the filter.
    """
    def f_smooth(t):
        t = np.asarray(t)
        t_min = t.min()
        t_max = t.max()
        t_samples = np.arange(t_min, t_max + sample_rate, sample_rate)
        y_samples = f(t_samples)
        y_smooth = savgol_filter(y_samples, window_length=window_length, polyorder=polyorder, mode='nearest')
        return np.interp(t, t_samples, y_smooth)
    return f_smooth
        
def continous_medfilt(f, kernel_size=11, sample_rate=0.001):
    """
    Return a smoothed version of the continuous function f using a median filter.
    The function f is sampled at intervals of sample_rate to apply the filter.
    """
    def f_smooth(t):
        t = np.asarray(t)
        t_min = t.min()
        t_max = t.max()
        t_samples = np.arange(t_min, t_max + sample_rate, sample_rate)
        y_samples = f(t_samples)
        y_smooth = medfilt(y_samples, kernel_size=kernel_size)
        return np.interp(t, t_samples, y_smooth)
    return f_smooth        

def unwrap_or_zero(f):
    if not f:
        return 0.0
    else:
        return f


### Thicket + Hatchet Integration
Convert the CSV datasets into thicket/hatchet objects for analysis

In [ ]:
def create_hatchet_dag(call_graph: pd.DataFrame, metric_functions: Dict[str, callable]):
    df = call_graph.copy()
    # Asume que viene “bien formado”: cada fila es una invocación con Depth que respeta el anidamiento.
    # Si no está ordenado por Start Time, ordénalo una vez.
    df = df.sort_values(by='Start Time')

    # Filtrado opcional por duración, replica tu lógica si quieres:
    df = df[df['Duration'] >= 0.001]

    pref3 = df['Name'].str[:3].str.lower()
    df = df[~pref3.isin(['hip', 'mpi'])]
    
    # Vectoriza cálculo de métricas por fila (Δ = f(end) - f(start))
    starts = df['Start Time'].to_numpy()
    ends   = df['End Time'].to_numpy()
    names  = df['Name'].to_numpy()
    depths = df['Depth'].to_numpy()
    durs   = df['Duration'].to_numpy()


    metrics_cols = {}
    for metric, f in tqdm(metric_functions.items()):
        try:
            vals = f(ends) - f(starts)            # ideal: f soporta arrays
            vals = np.asarray(vals, dtype=float)
            vals = np.nan_to_num(vals, nan=0.0, posinf=0.0, neginf=0.0)
        except Exception:
            # Fallback si f no vectoriza
            vals = np.fromiter(
                (unwrap_or_zero(f(e) - f(s)) for s, e in zip(starts, ends)),
                dtype=float, count=len(starts)
            )
        metrics_cols[metric] = vals

    # Construcción del DAG con una pila indexada por profundidad
    # Mantenemos las referencias a los últimos nodos por cada Depth.
    # Cuando Depth sube, el nuevo nodo se anexa como hijo del nodo en Depth-1.
    dag_roots: list = []
    last_at_depth: dict[int, dict] = {}

    # Usamos itertuples para bajo overhead
    M = len(df)
    for i, (name, depth, dur) in enumerate(zip(names, depths, durs)):
        node_metrics = {'time (inc)': float(dur), 'time': 0.0}
        for m, arr in metrics_cols.items():
            node_metrics[m] = float(arr[i])

        node = {'frame': {'name': name}, 'metrics': node_metrics, 'children': []}

        if depth == 0 or (depth - 1) not in last_at_depth:
            dag_roots.append(node)
        else:
            parent = last_at_depth[depth - 1]
            parent['children'].append(node)

        last_at_depth[depth] = node

        # Si retrocede la profundidad en la siguiente fila, iremos sobreescribiendo last_at_depth
        # No hace falta “cerrar” nada aún; calculamos 'time' más adelante en una segunda pasada.

    # Segunda pasada postorden para calcular 'time' = 'time (inc)' - suma hijos.
    # Podemos hacerlo con una DFS iterativa.
    stack = [(root, False) for root in dag_roots]
    while stack:
        node, visited = stack.pop()
        if not visited:
            stack.append((node, True))
            for ch in node['children']:
                stack.append((ch, False))
        else:
            child_incs = 0.0
            for ch in node['children']:
                child_incs += ch['metrics']['time (inc)']
            node['metrics']['time'] = node['metrics']['time (inc)'] - child_incs

    return dag_roots

# Hierarchical DataFrames

Organize the data into hierarchical dataframes for easier analysis:
- `Ensemble`: all the runs of a given configuration
- `Run`: a single run of a configuration
- `Node`: a single node in a run
- `Rank`: a single MPI rank in a node

In [ ]:
class Rank:
	def __init__(self, node: str, name: str, call_graph: pd.DataFrame):
		self.node = node
		self.name = name
		self.call_graph = call_graph
		assert self.call_graph.columns.tolist() == ['Thread', 'Group', 'Depth', 'Name', 'Start Time', 'End Time', 'Duration'], f"Call graph DataFrame has incorrect columns, found {call_graph.columns.tolist()}"
		
	def get_all_code_regions(self) -> List[str]:
		return self.call_graph['Name'].unique().tolist()

	def compute_attribution(
		self,
		metrics_accumulated: Dict[str, callable],
		duration_threshold: float = 0.001,
		weights_by_metric: Optional[Dict[str, np.ndarray]] = None):
		df = self.call_graph
		df = df[df['Duration'] >= duration_threshold]
		pref3 = df['Name'].str[:3].str.lower()
		df = df[~pref3.isin(['hip', 'mpi'])]

		if df.empty:
			return pd.DataFrame(columns=['Code Region'] + list(metrics_accumulated.keys()))

		starts = df['Start Time'].to_numpy()
		ends   = df['End Time'].to_numpy()
		names  = df['Name'].to_numpy()

		cols = {}
		for metric, f in metrics_accumulated.items():
			try:
				deltas = f(ends) - f(starts)
				deltas = np.asarray(deltas, dtype=float)
				deltas = np.nan_to_num(deltas, nan=0.0, posinf=0.0, neginf=0.0)
			except Exception:
				deltas = np.fromiter(
					(unwrap_or_zero(f(e) - f(s)) for s, e in zip(starts, ends)),
					dtype=float, count=len(starts)
				)

			# Apply weights if present for this metric (1.0 if not provided)
			if weights_by_metric and metric in weights_by_metric:
				w = np.asarray(weights_by_metric[metric], dtype=float)
				if w.shape[0] == deltas.shape[0]:
					deltas = deltas * w
				elif w.size == 1:
					deltas = deltas * float(w)
				else:
					# Safe fallback if the length does not match for some reason
					deltas = deltas * 1.0

			cols[metric] = deltas

		out = pd.DataFrame(cols)
		out['Code Region'] = names
		# Aggregate by region
		out = out.groupby('Code Region', as_index=False).sum()
		return out

	def refresh(self):
		return Rank(self.node, self.name, self.call_graph)

	def __str__(self):
		return f"Rank({self.name})"

def build_inclusive_masks_for_rank(breaks, s, e, names, depths):
	"""
	For each segment [breaks[i], breaks[i+1]), return a dict:
	  masks[region_name][i] = True  iff the region is on the call stack
								   (ancestor of top-of-stack) for the ENTIRE segment.
	Assumptions:
	  - breaks includes all start/end times (so inside a segment no boundary occurs).
	  - 'depths' is 0-based, increasing with nesting.
	"""
	seg_count = breaks.size - 1
	masks = defaultdict(lambda: np.zeros(seg_count, dtype=bool))

	lefts, rights = breaks[:-1], breaks[1:]
	for i in range(seg_count):
		L, R = lefts[i], rights[i]

		# Regions that cover the whole segment (active across the entire segment)
		idx = np.where((s <= L) & (e >= R))[0]
		if idx.size == 0:
			continue

		# Top of stack depth
		dmax = depths[idx].max()

		# Inclusive chain = exactly one region per depth 0..dmax that covers the whole segment
		# (there should typically be at most one per depth due to proper nesting)
		for d in range(dmax + 1):
			cand = idx[depths[idx] == d]
			if cand.size:
				# If instrumentation duplicates exist at same depth, pick the one that covers the segment;
				# here cand already covers segment; pick the first deterministically.
				j = cand[0]
				masks[names[j]][i] = True

	return masks

class Node:
	def __init__(self, name: str, metrics_df: pd.DataFrame, ranks: List[Rank]):
		self.name = name
		self.ranks = ranks
		self.metrics = metrics_df
		assert self.metrics.columns.tolist() == ['Group', 'Metric Name', 'Time', 'Value'], f"Metrics DataFrame has incorrect columns, found {metrics_df.columns.tolist()}"

	def refresh(self):
		return Node(self.name, self.metrics, [rank.refresh() for rank in self.ranks])

	def get_metric_names(self) -> List[str]:
		return self.metrics['Metric Name'].unique().tolist()
	
	def get_metric_samples(self, metric_name: str) -> pd.DataFrame:
		return self.metrics[self.metrics['Metric Name'] == metric_name]

	def _filtered_callgraph(self, rank: Rank, duration_threshold: float):
		df = rank.call_graph
		df = df[df['Duration'] >= duration_threshold]
		pref3 = df['Name'].str[:3].str.lower()
		df = df[~pref3.isin(['hip', 'mpi'])]
		# Finite and non-degenerate
		mfin = np.isfinite(df['Start Time']) & np.isfinite(df['End Time'])
		df = df[mfin & (df['End Time'] > df['Start Time'])]
		return df

	def compute_attribution(
		self,
		ranks_to_metrics: Dict[str, List[str]] = {},
		instantaneous_metrics = [],
		duration_threshold: float = 0.001,
	):
		# --- Only the metrics actually used ---
		used_metrics = sorted({m for ml in ranks_to_metrics.values() for m in ml})
		if not used_metrics:
			return pd.DataFrame(columns=['Code Region'])

		# accumulated f(t) per metric (if instantaneous, integrate first)
		metrics_continuous = {m: sampled_to_continuous(self.get_metric_samples(m)) for m in used_metrics}
		metrics_accumulated = {
			m: (numerical_integral(metrics_continuous[m]) if m in instantaneous_metrics else metrics_continuous[m])
			for m in used_metrics
		}
		# print(f"Node {self.name} has metrics: {list(metrics_accumulated.keys())}")

		# --- Pre-filtered by rank (and base arrays, incl. Depth) ---
		rank_df = {r.name: self._filtered_callgraph(r, duration_threshold) for r in self.ranks}
		rank_events = {
			rname: (
				df['Start Time'].to_numpy(),
				df['End Time'].to_numpy(),
				df['Name'].to_numpy(),
				df['Depth'].astype(int).to_numpy()
			)
			for rname, df in rank_df.items()
		}

		# metric -> ranks that use it
		metric_to_ranks: Dict[str, List[str]] = {}
		for rname, mlist in ranks_to_metrics.items():
			for m in mlist:
				metric_to_ranks.setdefault(m, []).append(rname)

		# Accumulator: per rank → per region → per metric → energy
		rank_tables: Dict[str, Dict[str, Dict[str, float]]] = {r.name: {} for r in self.ranks}

		# --- Loop per metric: segment time, split across ranks, and attribute inclusively ---
		for metric in used_metrics:
			f = metrics_accumulated[metric]
			rnames = [rn for rn in metric_to_ranks.get(metric, []) if rn in rank_events]
			if not rnames:
				continue

			# Global cuts (union of starts/ends of these ranks)
			all_s, all_e = [], []
			for rn in rnames:
				s, e, _, _ = rank_events[rn]
				if s.size:
					all_s.append(s); all_e.append(e)
			if not all_s:
				continue

			S_concat = np.concatenate(all_s)
			E_concat = np.concatenate(all_e)
			breaks = np.unique(np.concatenate([S_concat, E_concat]))
			breaks = breaks[np.isfinite(breaks)]
			if breaks.size < 2:
				continue
			seg_count = breaks.size - 1

			# ΔE per segment (sanitized)
			try:
				E_vals = f(breaks)
				E_vals = np.asarray(E_vals, dtype=float)
				if E_vals.shape != breaks.shape:
					raise ValueError
			except Exception:
				print(f"Warning: Non-vectorized metric function for metric {metric}, falling back to loop.")
				E_vals = np.array([float(f(float(t))) for t in breaks], dtype=float)

			dE = E_vals[1:] - E_vals[:-1]
			dE = np.nan_to_num(dE, nan=0.0, posinf=0.0, neginf=0.0)
			dE = np.maximum(dE, 0.0)

			# Which ranks are active in each segment?
			active_by_rank: Dict[str, np.ndarray] = {}
			lr_by_rank: Dict[str, Tuple[np.ndarray, np.ndarray]] = {}

			zeros_seg = np.zeros(seg_count, dtype=bool)
			for rn in rnames:
				s, e, _, _ = rank_events[rn]
				if s.size == 0:
					active_by_rank[rn] = zeros_seg
					lr_by_rank[rn] = (np.empty(0, dtype=int), np.empty(0, dtype=int))
					continue

				L = np.searchsorted(breaks, s, side='right') - 1
				R = np.searchsorted(breaks, e, side='left') - 1
				valid = (R >= L) & (R >= 0) & (L < seg_count)
				L = np.clip(L[valid], 0, seg_count - 1)
				R = np.clip(R[valid], 0, seg_count - 1)

				lr_by_rank[rn] = (L, R)

				# coverage count via difference array
				diff = np.zeros(seg_count + 1, dtype=int)
				# vectorized add/sub using slices
				# (loop remains but touches integers only; fast and cache-friendly)
				for l, r in zip(L, R):
					diff[l] += 1
					diff[r + 1] -= 1
				c = np.cumsum(diff[:-1])
				active_by_rank[rn] = (c > 0)

			# k(t): number of active ranks per segment
			k = np.zeros(seg_count, dtype=int)
			for rn in rnames:
				k += active_by_rank[rn].astype(int)

			# Energy share per segment for each rank: ΔE/k (if k==0, remains unattributed)
			share_per_seg = np.zeros_like(dE, dtype=float)
			mask_k = (k > 0)
			share_per_seg[mask_k] = dE[mask_k] / k[mask_k]

			# Inclusive attribution per rank: replicate share_per_seg to the entire inclusive chain
			for rn in rnames:
				s, e, names, depths = rank_events[rn]
				if s.size == 0:
					continue

				# Fast coverage masks per region using L/R (equivalent to inclusive chain under proper nesting):
				# region mask is True on segments where ANY of that region's events covers the whole segment.
				L, R = lr_by_rank[rn]
				masks: Dict[str, np.ndarray] = defaultdict(lambda: np.zeros(seg_count, dtype=bool))
				for j in range(L.size):
					l = L[j]; r = R[j]
					if r < l:
						continue
					masks[names[j]][l:r+1] = True

				# Add the SAME share_per_seg energy to each ancestor/region mask
				for region_name, seg_mask in masks.items():
					if not seg_mask.any():
						continue
					# sum over selected segments
					contrib = float(share_per_seg[seg_mask].sum())
					if contrib == 0.0:
						continue
					tbl = rank_tables[rn].setdefault(region_name, {})
					tbl[metric] = tbl.get(metric, 0.0) + contrib

		# --- Convert to per-rank DataFrames and do a wide merge ---
		attribution = None
		for rank in self.ranks:
			rn = rank.name
			# Filter only the metrics requested for this rank
			wanted = ranks_to_metrics.get(rn, [])
			if not wanted:
				continue

			rows = []
			for region, mdict in rank_tables.get(rn, {}).items():
				row = {'Code Region': region}
				# Ensure a column for each requested metric
				for m in wanted:
					row[m] = mdict.get(m, 0.0)
				rows.append(row)

			df_rank = pd.DataFrame(rows) if rows else pd.DataFrame({'Code Region': []})
			# Ensure columns if they are missing
			for m in wanted:
				if m not in df_rank.columns:
					df_rank[m] = 0.0

			# Rename metrics with rank suffix to avoid collisions
			rename_map = {m: f"{m}_{rn}" for m in wanted}
			df_rank = df_rank.rename(columns=rename_map)

			# Outer merge on 'Code Region'
			attribution = df_rank if attribution is None else pd.merge(
				attribution, df_rank, on='Code Region', how='outer'
			)

		# If there was nothing, return minimal header
		if attribution is None:
			all_cols = ['Code Region'] + [f"{m}_{r.name}" for r, ml in ranks_to_metrics.items() for m in ml]
			return pd.DataFrame(columns=all_cols)

		return attribution.fillna(0.0)

	def __str__(self):
		return f"Node(name={self.name}, ranks={[rank.name for rank in self.ranks]})"

class Run:
	def __init__(self, path: str, nodes: List[Node] = []):
		self.path = path
		self.nodes = nodes
		
	def refresh(self):
		return Run(self.path, [node.refresh() for node in self.nodes])

	@staticmethod
	def from_trace_path(trace_path: str, node_ranks: Dict[str, List[str]]):
		# Load the trace dataframes for each node and its ranks
		if not os.path.exists(trace_path):
			raise FileNotFoundError(f"Trace file {trace_path} does not exist.")
		# print(f"Loading trace from {trace_path} for idle period {idle_period_ms} ms and active period {active_period_ms} ms.")
		# trace_to_csv(trace_path, processes, METRICS_TO_ATTRIBUTE)
		# Load the CSV files into dataframes
		all_threads = []
		for _, ranks in node_ranks.items():
			all_threads.extend(ranks)
		result = {}
		trace_to_csv(trace_path, all_threads, METRICS_TO_ATTRIBUTE)
		for node, ranks in node_ranks.items():
			metrics_df = read_metrics_dataframe(f"{ranks[0]}_metrics.csv")
			print
			assert metrics_df.columns.tolist() == ['Group', 'Metric Name', 'Time', 'Value'], f"Metrics DataFrame has incorrect columns, found {metrics_df.columns.tolist()}"
			rank_callgraphs = []
			for rank in ranks:
				if os.path.exists(f"{rank}_Master_thread_callgraph.csv"):
					call_graph = read_call_graph_dataframe(f"{rank}_Master_thread_callgraph.csv")
					assert call_graph.columns.tolist() == ['Thread', 'Group', 'Depth', 'Name', 'Start Time', 'End Time', 'Duration'], f"Call graph DataFrame has incorrect columns, found {call_graph.columns.tolist()}"
					# Construct the Hatchet and Thicket graphs
					rank_callgraphs.append(Rank(node, rank, call_graph))
			result[node] = Node(node, metrics_df, rank_callgraphs)
		return Run(trace_path, list(result.values()))

	def compute_attribution(self, ranks_to_metrics: Dict[str, List[str]]={}, instantaneous_metrics=[]):
		attributions = {}
		for node in self.nodes:
			print(f"Computing attribution for node {node.name}...")
			attributions[node.name] = node.compute_attribution(ranks_to_metrics, instantaneous_metrics)
		return attributions
	
	def get_metric_samples(self, node_name: str, metric_name: str) -> pd.DataFrame:
		for node in self.nodes:
			if node.name == node_name:
				return node.get_metric_samples(metric_name)
		raise ValueError(f"Node {node_name} not found in run.")
	
	def to_thicket(self, **metadata) -> tt.Thicket:
		thickets = []
		
		for node in self.nodes:
			# self.get_metric_samples(node)
			metric_functions = {}
			for metric_name in node.get_metric_names():
				metric_functions[metric_name] = sampled_to_continuous(node.get_metric_samples(metric_name))
			
			for rank in node.ranks:
				print(f"Creating Thicket for node {node.name}, rank {rank.name}...")
				dag = create_hatchet_dag(rank.call_graph, metric_functions)
				tf = tt.Thicket.from_literal(dag)
				# tf = tt.Thicket.from_literal(dag)
				print("Thicket created from DAG.")
				tf.metadata = pd.DataFrame.from_dict(tf.profile_mapping, orient="index")
				tf.metadata['rank'] = rank.name
				tf.metadata['node'] = node.name
				tf.metadata = tf.metadata.assign(**metadata)
				tf.metadata.index.name = tf.dataframe.index.names[1]
		
				thickets.append(tf)
		
		print(f"Total thickets created: {len(thickets)}")
		if thickets:
			print("Combining thickets...")
			return tt.Thicket.concat_thickets(thickets, disable_tqdm=True)
		raise ValueError("No thickets were created from the run.")
		
	def __str__(self):
		return f"Run(path={self.path}, nodes={[str(node) for node in self.nodes]})"

class Ensemble:
	def __init__(self, runs: List[Run]):
		self.runs = runs
		self.attributions = {}
	
	@staticmethod
	def from_trace_paths(trace_paths: List[str], node_ranks: Dict[str, List[str]]):
		runs = []
		for path in trace_paths:
			print(f"Loading run from trace path: {path}")
			run = Run.from_trace_path(path, node_ranks)
			runs.append(run)
		return Ensemble(runs)
 
	def refresh(self):
		return Ensemble([run.refresh() for run in self.runs])
	
	def compute_attribution(self, ranks_to_metrics: Dict[str, List[str]]={}, instantaneous_metrics=[]):
		if self.attributions != {}:
			print("Attributions already computed, returning cached results.")
			return self.attributions
		
		for run in self.runs:
			self.attributions[os.path.basename(os.path.dirname(run.path))] = run.compute_attribution(ranks_to_metrics, instantaneous_metrics)
		return self.attributions
 
	def compute_attribution_across_runs(self, ranks_to_metrics: Dict[str, List[str]] = {}, instantaneous_metrics = []):
		attributions = self.compute_attribution(ranks_to_metrics, instantaneous_metrics)
		if not attributions:
			return pd.DataFrame(columns=['Run', 'Node', 'Code Region', 'Metric', 'Value'])

		records = []
		for run_name, node_attribs in attributions.items():
			for node_name, df in node_attribs.items():
				if df is None or df.empty: 
					continue
				code = df['Code Region'].to_numpy()
				for metric in df.columns:
					if metric == 'Code Region': 
						continue
					vals = df[metric].to_numpy()
					# Extend in bulk (no per-iteration concatenations)
					records.extend(
						{'Run': run_name, 'Node': node_name, 'Code Region': c, 'Metric': metric, 'Value': v}
						for c, v in zip(code, vals)
					)
		return pd.DataFrame.from_records(records)

	def to_thicket(self) -> tt.Thicket:
		thickets = []
		for run in self.runs:
			name = os.path.basename(os.path.dirname(run.path))
			print(f"Creating Thicket for run {run.path} ({name})...")
			tf = run.to_thicket(trace=run.path, run=name)
			thickets.append(tf)
		if thickets:
			return tt.Thicket.concat_thickets(thickets, disable_tqdm=True)
		else:
			return tt.Thicket()

In [ ]:
def take_alphanumeric(s):
    # Take up to the first non-alphanumeric characters
    match = re.match(r'^[~<, >:a-zA-Z0-9_]+', s)
    return match.group(0) if match else s

In [ ]:
# 3) Un solo PatchCollection con muchos rectángulos (¡una llamada de dibujo!)
from matplotlib.patches import Rectangle
from matplotlib.collections import PatchCollection

# First, compare the energy graphs over time for both runs
# Just the raw metrics, not attributions
def visualize_energy_of_runs(runs_fp: Ensemble, runs_mxp: Optional[Ensemble]=None, gpus: List[str]=['A2rocm_smi:::energy_count:device=0', 'A2rocm_smi:::energy_count:device=2', 'A2rocm_smi:::energy_count:device=4', 'A2rocm_smi:::energy_count:device=6'], name='', show_callgraph_events: str='full', run_count: int=1, duration_threshold: float=0.1):
    plt.figure(figsize=(16, 8))
    if name != '':
        name += ' '
        
    for i, run in enumerate(runs_fp.runs):
        if i >= run_count:
            break
        for j, gpu in enumerate(gpus):
            run_node0_energy = run.get_metric_samples('node0', gpu)
            # run_node0_energy['Value'] = run_node0_energy['Value'] - run_node0_energy['Value'].min()
            # Pandas doesnt like assigning to a series so we do it this way
            run_node0_energy.loc[:, 'Value'] = run_node0_energy['Value'] - run_node0_energy['Value'].min()
            # plt.plot(run_node0_energy['Time'], run_node0_energy['Value'], label=f'{name}Full Precision' if i + j == 0 else '', color='red', lw=1)
            if runs_mxp is not None:
                plt.plot(run_node0_energy['Time'], run_node0_energy['Value'], label=f'{name}Full Precision' if i + j == 0 else '', color='red', lw=1)
            else:
                plt.plot(run_node0_energy['Time'], run_node0_energy['Value'], label=name if i + j == 0 else '', color='red', lw=1)

    if runs_mxp is not None:
        for i, run in enumerate(runs_mxp.runs):
            if i >= run_count:
                break
            for j, gpu in enumerate(gpus):
                run_mxp_node0_energy = run.get_metric_samples('node0', gpu)
                # run_mxp_node0_energy['Value'] = run_mxp_node0_energy['Value'] - run_mxp_node0_energy['Value'].min()
                run_mxp_node0_energy.loc[:, 'Value'] = run_mxp_node0_energy['Value'] - run_mxp_node0_energy['Value'].min()
                plt.plot(run_mxp_node0_energy['Time'], run_mxp_node0_energy['Value'], label=f'{name}Mixed Precision' if i + j == 0 else '', color='blue', lw=1)

    # 2) Prepara colores por categoría (sin dict ni set por evento)
    if show_callgraph_events:
        first_run = runs_fp.runs[0]
        if show_callgraph_events == 'mxp' and runs_mxp is not None:
            first_run = runs_mxp.runs[0]
        ax = plt.gca()
        call_graph = first_run.nodes[0].ranks[0].call_graph
        call_graph = call_graph.sort_values(by='Start Time')
        call_graph = call_graph[call_graph['Duration'] >= duration_threshold]  # Filter out very short events
        call_graph.loc[:, 'Name'] = call_graph['Name'].apply(take_alphanumeric)
        
        df = call_graph.copy()
        names = pd.Categorical(df['Name'])
        ncat = len(names.categories)
        cmap = plt.get_cmap('tab10', max(10, ncat))
        color_arr = np.asarray([cmap(i) for i in names.codes])

        y0, y1 = ax.get_ylim()
        h = (y1 - y0)

        starts = df['Start Time'].to_numpy()
        ends   = df['End Time'  ].to_numpy()
        widths = ends - starts

        rects = [Rectangle((x, y0), w, h) for x, w in zip(starts, widths)]
        pc = PatchCollection(rects, facecolors=color_arr, edgecolor='none', alpha=0.1, rasterized=True, zorder=-5)
        ax.add_collection(pc)

        # 4) Etiquetas: 1 por región (no por evento) y solo para las más importantes
        #    Calculamos duración total por Name y etiquetamos las top-K.
        K = 8  # ajusta según gustes
        totals = df.groupby(df['Name'])['Duration'].sum().nlargest(K)

        # posición de etiqueta = centro del primer intervalo de esa región (o del intervalo mediano)
        for region in totals.index:
            sub = df.loc[df['Name'] == region]
            # usa el primer intervalo largo para colocar etiqueta
            i = sub['Duration'].to_numpy().argmax()
            sx = sub['Start Time'].iloc[i]
            ex = sub['End Time'  ].iloc[i]
            xmid = 0.5*(sx+ex)
            ax.text(xmid, y0 + 0.95*h, region, ha='center', va='top', fontsize=8, rotation=90,
                    clip_on=True)
        
    plt.xlabel('Time (s)')
    plt.ylabel('Energy Count (Joules)')
    if name != '':
        plt.title(f'{name}Energy Consumption Over Time')
    else:
        plt.title('Energy Consumption Over Time')
    plt.legend()
    plt.show()


In [ ]:
def visualize_power_of_runs(runs_fp: Ensemble, runs_mxp: Optional[Ensemble]=None, gpus: List[str]=['A2rocm_smi:::energy_count:device=0', 'A2rocm_smi:::energy_count:device=2', 'A2rocm_smi:::energy_count:device=4', 'A2rocm_smi:::energy_count:device=6'], differentiate_rocm=True, name='', show_callgraph_events: str='full', run_count=1, duration_threshold: float=0.1):
    plt.figure(figsize=(24, 8))
    if name != '':
        name += ' '
        
    for i, run in enumerate(runs_fp.runs):
        if i >= run_count:
            break
        for j, gpu in enumerate(gpus):
            run_node0_metrics = run.get_metric_samples('node0', gpu)
            # If this is rocm, differentiate by time delta
            if differentiate_rocm and 'rocm' in gpu.lower():
                # run_node0_metrics['Value'] = run_node0_metrics['Value'] - run_node0_metrics['Value'].min()
                run_node0_metrics.loc[:, 'Value'] = run_node0_metrics['Value'] - run_node0_metrics['Value'].min()
                run_node0_energy = sampled_to_continuous(run_node0_metrics)
                run_node0_power = numerical_derivative(run_node0_energy)
                if runs_mxp is not None:
                    plt.plot(run_node0_metrics['Time'], run_node0_power(run_node0_metrics['Time']), label=f'{name}Full Precision' if i + j == 0 else '', color='red', lw=1)
                else:
                    plt.plot(run_node0_metrics['Time'], run_node0_power(run_node0_metrics['Time']), label=name if i + j == 0 else '', color='red', lw=1)
                
                # run_node0_metrics = run_node0_metrics[run_node0_metrics['Value'].diff() > 0]
                # time_deltas = run_node0_metrics['Time'].diff()
                # energy_deltas = run_node0_metrics['Value'].diff()
                # power_values = energy_deltas / time_deltas
                # run_node0_metrics['Value'] = power_values
                # plt.plot(run_node0_metrics['Time'], run_node0_metrics['Value'], label=f'{name}Full Precision' if i + j == 0 else '', color='red', lw=1)
            else:
                plt.plot(run_node0_metrics['Time'], run_node0_metrics['Value'], label=f'{name}Full Precision' if i + j == 0 else '', color='red', lw=1)

    if runs_mxp is not None:
        for i, run in enumerate(runs_mxp.runs):
            if i >= run_count:
                break
            for j, gpu in enumerate(gpus):
                run_node0_metrics = run.get_metric_samples('node0', gpu)
                # If this is rocm, differentiate by time delta
                if differentiate_rocm and 'rocm' in gpu.lower():
                    # run_node0_metrics['Value'] = run_node0_metrics['Value'] - run_node0_metrics['Value'].min()
                    run_node0_metrics.loc[:, 'Value'] = run_node0_metrics['Value'] - run_node0_metrics['Value'].min()
                    run_node0_energy = sampled_to_continuous(run_node0_metrics)
                    run_node0_power = numerical_derivative(run_node0_energy)
                    plt.plot(run_node0_metrics['Time'], run_node0_power(run_node0_metrics['Time']), label=f'{name}Mixed Precision' if i + j == 0 else '', color='blue', lw=1)
                    
                    # run_node0_metrics = run_node0_metrics[run_node0_metrics['Value'].diff() > 0]
                    # time_deltas = run_node0_metrics['Time'].diff()
                    # energy_deltas = run_node0_metrics['Value'].diff()
                    # power_values = energy_deltas / time_deltas
                    # run_node0_metrics['Value'] = power_values
                    # plt.plot(run_node0_metrics['Time'], run_node0_metrics['Value'], label=f'{name}Mixed Precision' if i + j == 0 else '', color='blue', lw=1)
                else:
                    plt.plot(run_node0_metrics['Time'], run_node0_metrics['Value'], label=f'{name}Mixed Precision' if i + j == 0 else '', color='blue', lw=1)

    # 2) Prepara colores por categoría (sin dict ni set por evento)
    if show_callgraph_events:
        first_run = runs_fp.runs[0]
        if show_callgraph_events == 'mxp' and runs_mxp is not None:
            first_run = runs_mxp.runs[0]
        ax = plt.gca()
        call_graph = first_run.nodes[0].ranks[0].call_graph
        call_graph = call_graph.sort_values(by='Start Time')
        call_graph = call_graph[call_graph['Duration'] >= duration_threshold]  # Filter out very short events
        call_graph.loc[:, 'Name'] = call_graph['Name'].apply(take_alphanumeric)
        df = call_graph.copy()
        names = pd.Categorical(df['Name'])
        ncat = len(names.categories)
        cmap = plt.get_cmap('tab10', max(10, ncat))
        color_arr = np.asarray([cmap(i) for i in names.codes])

        y0, y1 = ax.get_ylim()
        h = (y1 - y0)

        starts = df['Start Time'].to_numpy()
        ends   = df['End Time'  ].to_numpy()
        widths = ends - starts

        rects = [Rectangle((x, y0), w, h) for x, w in zip(starts, widths)]
        pc = PatchCollection(rects, facecolors=color_arr, edgecolor='none', alpha=0.1, rasterized=True, zorder=-5)
        ax.add_collection(pc)

        K = 8  # ajusta según gustes
        totals = df.groupby(df['Name'])['Duration'].sum().nlargest(K)

        # posición de etiqueta = centro del primer intervalo de esa región (o del intervalo mediano)
        for region in totals.index:
            sub = df.loc[df['Name'] == region]
            # usa el primer intervalo largo para colocar etiqueta
            i = sub['Duration'].to_numpy().argmax()
            sx = sub['Start Time'].iloc[i]
            ex = sub['End Time'  ].iloc[i]
            xmid = 0.5*(sx+ex)
            ax.text(xmid, y0 + 0.95*h, region, ha='center', va='top', fontsize=8, rotation=90,
                    clip_on=True)
        
    plt.xlabel('Time (s)')
    plt.ylabel('Energy Count (Watts)')
    if name != '':
        plt.title(f'{name}Energy Consumption Over Time')
    else:
        plt.title('Energy Consumption Over Time')
    plt.tight_layout()
    plt.legend()
    plt.show()


In [ ]:
# First, replace all function names with the following function:
def replace_prefix_and_suffix(name: str) -> str:
    """Replace the prefix and everything after the `<` or `(` in the function name."""
    # name = name.replace('HPLMXP_', 'HPL_')  # Replace HPLMXP with HPL
    if '<' in name:
        name = name.split('<')[0]
    if '(' in name:
        name = name.split('(')[0]
    return name.strip()


def visualize_code_region_energy_table(ranks_to_metrics: Dict[str, List[str]], runs_fp: Ensemble, runs_mxp: Optional[Ensemble]=None, title: str='Energy Heatmap', differentiate: bool=False, figsize: Tuple[int, int]=(12, 8), fp_prefix: str="", mxp_prefix: str="", max_count=None):
    runs_fp = runs_fp.refresh()
    if runs_mxp is not None:
        runs_mxp = runs_mxp.refresh()
    run_attributions = runs_fp.compute_attribution_across_runs(ranks_to_metrics)
    # Average all the mxp runs and non-mxp runs each
    fp_avg = run_attributions.copy()
    fp_avg['Code Region'] = fp_avg['Code Region'].apply(take_alphanumeric)
    fp_avg = fp_avg.groupby(['Node', 'Code Region', 'Metric'], as_index=False)['Value'].mean()
    fp_avg['Run'] = 'Full'
    # Sort the dataframe by value
    fp_avg = fp_avg.sort_values(by='Value', ascending=False)
    if fp_prefix != "":
        fp_avg['Code Region'] = fp_avg['Code Region'].apply(lambda x: x.replace(fp_prefix, ''))

    mxp_avg = pd.DataFrame(columns=fp_avg.columns)
    if runs_mxp is not None:
        run_attributions = runs_mxp.compute_attribution_across_runs(ranks_to_metrics)
        mxp_avg = run_attributions.copy()
        mxp_avg = mxp_avg.groupby(['Node', 'Code Region', 'Metric'], as_index=False)['Value'].mean()
        mxp_avg['Code Region'] = mxp_avg['Code Region'].apply(take_alphanumeric)
        if mxp_prefix != "":
            mxp_avg['Code Region'] = mxp_avg['Code Region'].apply(lambda x: x.replace(mxp_prefix, ''))
    mxp_avg['Run'] = 'MxP'
    heatmap_data = pd.concat([fp_avg, mxp_avg])
    heatmap_data = heatmap_data[heatmap_data['Value'] > 3]
    heatmap_data = heatmap_data.pivot_table(index='Code Region', columns=['Run', 'Metric'], values='Value', fill_value=0.0, sort=True)

    if max_count is not None:
        # Take the top max_count code regions by total energy
        which_to_keep = heatmap_data.sum(axis=1).sort_values(ascending=False).head(max_count).index
        heatmap_data = heatmap_data.loc[which_to_keep]
    else:
        # Still sort by total energy
        heatmap_data = heatmap_data.loc[heatmap_data.sum(axis=1).sort_values(ascending=False).index]

    # Show the heatmap
    plt.figure(figsize=(16, 12))
    sns.heatmap(heatmap_data, annot=True, fmt=".1f", cmap="coolwarm")
    plt.title(title)
    plt.xlabel("Run Type")
    plt.xticks(rotation=90, ha='right')
    plt.ylabel("Code Region")
    plt.show()

# Begin Analysis!
Load in the rocHPL data

In [ ]:
double_gcd_runs = {
    (time_active, time_idle): Ensemble.from_trace_paths(
        DOUBLE_GCD_TRACES[(time_active, time_idle)],
        DOUBLE_GCD_NODE_RANKS
    )
    for (time_active, time_idle) in DOUBLE_GCD_TRACES.keys()
}

In [ ]:
single_gcd_runs = {
    (time_active, time_idle): Ensemble.from_trace_paths(
        SINGLE_GCD_TRACES[(time_active, time_idle)],
        SINGLE_GCD_NODE_RANKS
    )
    for (time_active, time_idle) in SINGLE_GCD_TRACES.keys()
}

# Compare HPL full and mixed precision implementations


In [ ]:
for time_idle in [400]:
    for time_active in ACTIVE_PERIODS_MS:
        visualize_power_of_runs(
            double_gcd_runs[(time_active, time_idle)],
            single_gcd_runs[(time_active, time_idle)],
            name=f'Double GCD Active {time_active}ms Idle {time_idle}ms',
            duration_threshold=0.0001,
            gpus=['A2rocm_smi:::energy_count:device=0']
        )

# rocHPL vs. rocHPL-MxP Energy Heatmap
The graph below shows the average energy attribution across each HPL run and HPL-MxP run side by side.

In [ ]:
def calculate_mean_absolute_percentage_error_between_metrics(runs: Run, metric_a: str, metric_b: str, differentiate_rocm: bool=True, graph_metrics: bool=False, t_start=None, t_end=None, sample_period=0.002) -> float:
    # Get the metric streams for both metrics
    samples_a = runs.get_metric_samples('node0', metric_a)
    samples_a.loc[:, 'Value'] = samples_a['Value'].fillna(1e-9)
    f_a = sampled_to_continuous(samples_a)
    if 'rocm' in metric_a.lower() and differentiate_rocm:
        # print("Differentiating ROCm metric A")
        f_a = numerical_derivative(f_a)
    samples_b = runs.get_metric_samples('node0', metric_b)
    samples_b.loc[:, 'Value'] = samples_b['Value'].fillna(1e-9)
    f_b = sampled_to_continuous(samples_b)
    if 'rocm' in metric_b.lower() and differentiate_rocm:
        # print("Differentiating ROCm metric B")
        f_b = numerical_derivative(f_b)
    # Use 3-12 seconds as the common time range
    if t_end is None:
        t_end = min([12.0, samples_a['Time'].max(), samples_b['Time'].max()])
    if t_start is None:
        t_start = min([3.0, max(t_end - 1.0, 0)])
    
    f_a = continuous_savgol_filter(f_a, window_length=51, polyorder=3)
    f_b = continuous_savgol_filter(f_b, window_length=51, polyorder=3)
    
    # Use a sample period of ~2ms
    sample_times = np.arange(t_start, t_end, step=sample_period)
    if graph_metrics:
        # Graph f_b and f_a to verify
        plt.figure(figsize=(12, 6))
        plt.plot(sample_times, f_a(sample_times), label=metric_a, color='red')
        plt.plot(sample_times, f_b(sample_times), label=metric_b, color='blue')
        error = np.abs(f_a(sample_times) - f_b(sample_times))
        plt.plot(sample_times, error, label='Absolute Error', color='green', alpha=0.5)
        # Plot an average line
        avg_a = np.mean(f_a(sample_times))
        avg_b = np.mean(f_b(sample_times))
        plt.axhline(y=avg_a, color='red', linestyle='--', label=f'{metric_a} Mean ({avg_a:.2f})')
        plt.axhline(y=avg_b, color='blue', linestyle='--', label=f'{metric_b} Mean ({avg_b:.2f})')
        plt.axhline(y=np.mean(error), color='green', linestyle='--', label=f'Mean Absolute Error ({np.mean(error):.2f})')
        plt.xlabel('Time (s)')
        plt.ylabel('Metric Value')
        plt.title(f'Comparison of {metric_a} and {metric_b}')
        plt.legend()
        plt.tight_layout()
        plt.show()

    # Compute the mean absolute percentage error
    mape = np.mean(np.abs((f_a(sample_times) - f_b(sample_times)) / f_b(sample_times))) * 100
    return mape

calculate_mean_absolute_percentage_error_between_metrics(double_gcd_runs[(1000, 1000)].runs[0], 'A2rocm_smi:::energy_count:device=0', 'A2coretemp:::craypm:accel0_power', graph_metrics=True)
calculate_mean_absolute_percentage_error_between_metrics(single_gcd_runs[(1000, 1000)].runs[0], 'A2rocm_smi:::energy_count:device=0', 'A2coretemp:::craypm:accel0_power', graph_metrics=True)
# calculate_mean_absolute_percentage_error_between_metrics(apu_runs[(1000, 1000)].runs[0], 'A2rocm_smi:::energy_count:device=0', 'A2coretemp:::craypm:accel0_power', graph_metrics=True)

In [ ]:
def calculate_correlation_between_metrics(runs: Run, metric_a: str, metric_b: str, differentiate_rocm: bool=True, graph_metrics: bool=False, method='pearson', t_start=None, t_end=None, sample_period=0.001) -> float:
    # Get the metric streams for both metrics
    samples_a = runs.get_metric_samples('node0', metric_a)
    samples_a.loc[:, 'Value'] = samples_a['Value'].fillna(1e-6)
    f_a = sampled_to_continuous(samples_a)
    if 'rocm' in metric_a.lower() and differentiate_rocm:
        # print("Differentiating ROCm metric A")
        f_a = numerical_derivative(f_a)
    samples_b = runs.get_metric_samples('node0', metric_b)
    samples_b.loc[:, 'Value'] = samples_b['Value'].fillna(1e-6)
    f_b = sampled_to_continuous(samples_b)
    if 'rocm' in metric_b.lower() and differentiate_rocm:
        # print("Differentiating ROCm metric B")
        f_b = numerical_derivative(f_b)
    f_a = continous_medfilt(f_a, kernel_size=51)
    f_b = continous_medfilt(f_b, kernel_size=51)
    f_a = continuous_savgol_filter(f_a, window_length=51, polyorder=3)
    f_b = continuous_savgol_filter(f_b, window_length=51, polyorder=3)
    # Use 3-12 seconds as the common time range
    if t_end is None:
        t_end = min([15.0, samples_a['Time'].max(), samples_b['Time'].max()])
    if t_start is None:
        t_start = min([2.0, max(t_end - 1.0, 0)])
        
    # Use a sample period of ~2ms
    sample_times = np.arange(t_start, t_end, step=sample_period)
    if graph_metrics:
        # Graph f_b and f_a to verify
        plt.figure(figsize=(12, 6))
        plt.plot(sample_times, f_a(sample_times), label=metric_a, color='red')
        plt.plot(sample_times, f_b(sample_times), label=metric_b, color='blue')
        error = np.abs(f_a(sample_times) - f_b(sample_times))
        plt.plot(sample_times, error, label='Absolute Error', color='green', alpha=0.5)
        # Plot an average line
        avg_a = np.mean(f_a(sample_times))
        avg_b = np.mean(f_b(sample_times))
        plt.axhline(y=avg_a, color='red', linestyle='--', label=f'{metric_a} Mean ({avg_a:.2f})')
        plt.axhline(y=avg_b, color='blue', linestyle='--', label=f'{metric_b} Mean ({avg_b:.2f})')
        plt.axhline(y=np.mean(error), color='green', linestyle='--', label=f'Mean Absolute Error ({np.mean(error):.2f})')
        plt.xlabel('Time (s)')
        plt.ylabel('Metric Value')
        plt.title(f'Comparison of {metric_a} and {metric_b}')
        plt.legend()
        plt.tight_layout()
        plt.show()

    def spearmanr(x, y):
        """Compute Spearman's rank correlation coefficient."""
        n = len(x)
        rank_x = np.argsort(np.argsort(x))
        rank_y = np.argsort(np.argsort(y))
        d = rank_x - rank_y
        d_squared = d ** 2
        numerator = 6 * np.sum(d_squared)
        denominator = n * (n**2 - 1)
        rho = 1 - (numerator / denominator)
        return rho, None

    # Compute the correlation coefficient
    if method == 'pearson':
        if np.std(f_a(sample_times)) == 0 or np.std(f_b(sample_times)) == 0:
            print("One of the metrics has zero standard deviation, correlation is undefined.")
            return 0.0
        corr = np.corrcoef(f_a(sample_times), f_b(sample_times))[0, 1]
    elif method == 'spearman':
        corr, _ = spearmanr(f_a(sample_times), f_b(sample_times))
    elif method == 'kendall':
        corr, _ = kendalltau(f_a(sample_times), f_b(sample_times))
    else:
        raise ValueError(f"Unknown correlation method: {method}")
    return corr

calculate_correlation_between_metrics(double_gcd_runs[(1000, 1000)].runs[0], 'A2rocm_smi:::energy_count:device=0', 'A2coretemp:::craypm:accel0_power', graph_metrics=True)
calculate_correlation_between_metrics(single_gcd_runs[(1000, 1000)].runs[0], 'A2rocm_smi:::energy_count:device=0', 'A2coretemp:::craypm:accel0_power', graph_metrics=True)
# calculate_correlation_between_metrics(apu_runs[(1000, 1000)].runs[0], 'A2rocm_smi:::energy_count:device=0', 'A2coretemp:::craypm:accel0_power', graph_metrics=True)

In [ ]:
def visualize_mape_heatmap(runs: Dict[Tuple[int, int], Ensemble], IDLE_PERIODS_MS: List[int], ACTIVE_PERIODS_MS: List[int], metric_a: str='A2rocm_smi:::energy_count:device=0', metric_b: str='A2coretemp:::craypm:accel0_power', sample_period: float=0.002, t_start: Optional[float]=None, t_end: Optional[float]=None):
    # Create a heatmap for all the idle-active period combinations
    mape_records = []
    for time_idle in IDLE_PERIODS_MS:
        for time_active in ACTIVE_PERIODS_MS:
            run = runs[(time_active, time_idle)].runs[0]
            mape = calculate_mean_absolute_percentage_error_between_metrics(
                run,
                metric_a,
                metric_b,
                # graph_metrics=False,
                sample_period=sample_period,
                t_start=t_start,
                t_end=t_end,
                graph_metrics=True,
            )
            mape_records.append({
                'Active Period (ms)': time_active,
                'Idle Period (ms)': time_idle,
                'MAPE (%)': mape
            })
            
    mape_df = pd.DataFrame.from_records(mape_records)
    mape_pivot = mape_df.pivot(index='Idle Period (ms)', columns='Active Period (ms)', values='MAPE (%)')
    # Make sure (1, 1) is at the bottom left
    mape_pivot = mape_pivot.sort_index(ascending=False)
    plt.figure(figsize=(12, 8))
    sns.heatmap(mape_pivot, annot=True, fmt=".2f", cmap="YlGnBu")
    plt.title('Mean Absolute Percentage Error (MAPE) between Power Metrics')
    plt.xlabel('Active Period (ms)')
    plt.ylabel('Idle Period (ms)')
    plt.show()

print("Visualizing MAPE Heatmap for 2 GCDs per GPU...")
visualize_mape_heatmap(double_gcd_runs, IDLE_PERIODS_MS, ACTIVE_PERIODS_MS)
print("Visualizing MAPE Heatmap for 1 GCD per GPU...")
visualize_mape_heatmap(single_gcd_runs, IDLE_PERIODS_MS, ACTIVE_PERIODS_MS)
# visualize_mape_heatmap(double_gcd_runs, [1, 1000], ACTIVE_PERIODS_MS)

In [ ]:
def visualize_correlation_heatmap(runs: Dict[Tuple[int, int], Ensemble], IDLE_PERIODS_MS: List[int], ACTIVE_PERIODS_MS: List[int], metric_a: str='A2rocm_smi:::energy_count:device=0', metric_b: str='A2coretemp:::craypm:accel0_power', method='pearson', sample_period: float=0.002, t_start: Optional[float]=None, t_end: Optional[float]=None):
    # Create a heatmap for all the idle-active period combinations
    corr_records = []
    for time_idle in IDLE_PERIODS_MS:
        for time_active in ACTIVE_PERIODS_MS:
            run = runs[(time_active, time_idle)].runs[0]
            corr = calculate_correlation_between_metrics(
                run,
                metric_a,
                metric_b,
                graph_metrics=False,
                method=method,
                sample_period=0.1,
            )
            # if abs(corr) < 0.0001:
            #     # Show the graph
            #     calculate_correlation_between_metrics(
            #         run,
            #         metric_a,
            #         metric_b,
            #         graph_metrics=True,
            #         method=method,
            #         sample_period=sample_period,
            #         t_start=t_start,
            #         t_end=t_end
            #     )
            corr_records.append({
                'Active Period (ms)': time_active,
                'Idle Period (ms)': time_idle,
                'Correlation': corr
            })

    corr_df = pd.DataFrame.from_records(corr_records)
    corr_pivot = corr_df.pivot(index='Idle Period (ms)', columns='Active Period (ms)', values='Correlation')
    # Make sure (1, 1) is at the bottom left
    corr_pivot = corr_pivot.sort_index(ascending=False)
    plt.figure(figsize=(12, 8))
    sns.heatmap(corr_pivot, annot=True, fmt=".2f", cmap="YlGnBu")
    plt.title(f'{method.title()} Correlation between Power Metrics')
    plt.xlabel('Active Period (ms)')
    plt.ylabel('Idle Period (ms)')
    plt.show()

print("Visualizing Correlation Heatmap for 2 GCDs per GPU...")
visualize_correlation_heatmap(double_gcd_runs, IDLE_PERIODS_MS, ACTIVE_PERIODS_MS)
print("Visualizing Correlation Heatmap for 1 GCD per GPU...")
visualize_correlation_heatmap(single_gcd_runs, IDLE_PERIODS_MS, ACTIVE_PERIODS_MS)
# visualize_correlation_heatmap(double_gcd_runs, [1, 1000], ACTIVE_PERIODS_MS)

In [ ]:
def calculate_power_functions(df: pd.DataFrame, differentiate_rocm: bool=True, keys: List[str]=None) -> Dict[str, Callable[[np.ndarray], np.ndarray]]:
    if keys is None:
        keys = df['Metric Name'].unique()
    else:
        keys = [key for key in keys if key in df['Metric Name'].unique()]
    power_functions = {}
    for metric_name in keys:
        metric_df = df[df['Metric Name'] == metric_name]
        f_metric = sampled_to_continuous(metric_df)
        if 'rocm' in metric_name.lower() and differentiate_rocm:
            power_functions[metric_name] = numerical_derivative(f_metric)
        else:
            power_functions[metric_name] = f_metric
    return power_functions

In [ ]:
def calculate_rising_and_falling_events(metrics: pd.DataFrame, call_graph: Optional[pd.DataFrame]=None, t_min=0, t_max=15, vt_min=0, vt_max=15,
                              key='A2rocm_smi:::energy_count:device=0',
                              fs=1_000,
                              median_win=21,
                              sg_win=11, sg_poly=1,
                              alpha_up=0.35, beta_dn=0.65,
                              band_frac_low=0.10,   # ±10% for idle band width (relative)
                              band_frac_high=0.08,  # ±8%  for active band width (relative)
                              band_min_watts=32.0,  # absolute minimum band half-width in watts
                              hold_time=0.010,
                              min_gap=0.020,
                              debug_second_pass=True, plot: bool=True, summary: bool=True, differentiate_rocm=True) -> List[Dict[str, Any]]:
    """
    Plot power vs. time, detect phase transitions (idle<->active), and measure
    transition durations using a robust, derivative-free method.

    ---------------------------
    What this function assumes:
    ---------------------------
    - The signal is a "spiky sawtooth" around two quasi-stable plateaus:
        * a lower 'idle' level
        * a higher 'active' level
      Short spikes/outliers are common and should *not* define the plateaus.
    - Transitions between plateaus are not instantaneous; we want the time from:
        t_start : when we *commit* to leaving the old plateau
        t_end   : when we *settle* inside a band around the new plateau and remain
                  there for a minimum hold time.

    ----------------------------------
    High-level algorithm (step-by-step)
    ----------------------------------
    1) Resample the provided power function uniformly on [t_min, t_max] at
       nominal sampling rate `fs`. This creates arrays `x` (time) and `y` (power).

    2) Robust smoothing:
       - Median filter (window `median_win`) removes narrow spikes without
         smearing step edges too much.
       - Savitzky–Golay filter (window `sg_win`, poly `sg_poly`) smooths residual
         noise while preserving step/edge shape better than a simple moving average.

    3) Robust level estimation:
       - Percentile clip the smoothed series to [1st, 99th] to suppress extreme
         outliers that would bias clustering.
       - Run K-means with k=2 on the clipped series to separate samples into
         two clusters. For each cluster, use the *median* (not the mean) as the
         representative level. Sort them to get:
             mu_low  ~ idle level
             mu_high ~ active level

    4) Build settling bands around those levels:
       - Compute a robust spread using MAD (Median Absolute Deviation) and scale
         by 1.4826 ~ sigma for Gaussian. The settling band half-width is the max of:
             (relative fraction of the center level) vs (absolute minimum watts)
             vs (MAD-based width)
         This ensures the band is wide enough to cover the sawtooth ripple but
         not so wide that it swallows a whole transition.

    5) Hysteresis thresholds to avoid chatter:
       - Two thresholds between mu_low and mu_high:
            th_up = mu_low + alpha_up * (mu_high - mu_low)   (enter ACTIVE)
            th_dn = mu_low + beta_dn  * (mu_high - mu_low)   (enter IDLE)
         with beta_dn < alpha_up. This prevents rapid bouncing due to ripple.

    6) Build a binary state machine over time:
       - Start in state=0 (idle).
       - If state==0 and y >= th_up  -> state=1 (we are rising into active).
       - If state==1 and y <= th_dn  -> state=0 (we are falling into idle).
       This tracks when we *intend* to be in each phase, ignoring brief crossings.

    7) For each 0->1 (rise) or 1->0 (fall) edge in the state machine:
       - Define t_start at the first sample where the state flips.
       - Search forward for the earliest index where the signal stays inside the
         corresponding target band (high band for rise, low band for fall) for
         at least `hold_time`. That index defines t_end.
       - Record event type ('rise' or 'fall'), t_start, t_end, and duration.

    8) Optional "second pass" if no events were found:
       - As a safety valve, widen the bands slightly and reduce hold time,
         then retry the detection (helps when the bands ended up too tight).

    9) Plot:
       - Raw power, smoothed power, hysteresis thresholds, settling bands
         (shaded), vertical lines at t_start / t_end, and shaded spans showing
         transition durations.
       - Overlay HIP/MPI call markers if present in `traces[TRACE][1]`.

    -----------------
    Key parameters:
    -----------------
    - fs:         resampling rate (Hz) used to discretize time.
    - median_win: median filter window length (odd). Choose ~1–2 sawtooth periods.
    - sg_win:     Savitzky–Golay window length (odd). Keep small to preserve steps.
    - sg_poly:    Savitzky–Golay polynomial degree (2 is usually enough).
    - alpha_up, beta_dn: hysteresis fractions between the two centroid levels.
    - 
    /high: relative half-band width (as a fraction of level).
    - band_min_watts: absolute minimum half-band width in watts.
    - hold_time:  required time the signal must remain in-band to consider it
                  "settled" at the new level (prevents false positives).
    - min_gap:    skip ahead by this time after closing an event to avoid counting
                  overlapping/duplicate transitions.
    - debug_second_pass: whether to widen bands & reduce hold time if nothing was found.

    Returns:
        events: list of dicts with fields:
            - type: 'rise' or 'fall'
            - t_start: transition start time (s)
            - t_end:   transition end time (s)
            - duration: t_end - t_start (s)
    """
    power_functions = calculate_power_functions(metrics, differentiate_rocm=differentiate_rocm, keys=[key])
    
    # --- 1) Uniform resampling over [t_min, t_max] ---
    x = np.linspace(t_min, t_max, num=max(2, int((t_max - t_min) * fs)))
    ref_power = power_functions.get(key, None)
    if ref_power is None:
        raise KeyError(f"Series '{key}' not found in power_functions")
    y = ref_power(x)

    # --- 2) Robust smoothing: Median -> Savitzky–Golay ---
    # Both median and Savitzky–Golay require odd window sizes. If an even value
    # is provided, bump it by 1 so the filters run as expected.
    if median_win % 2 == 0: median_win += 1
    if sg_win % 2 == 0: sg_win += 1
    # Median filter: strongly suppresses narrow spikes without blurring edges much.
    # y_med = medfilt(y, kernel_size=median_win)
    # Savitzky–Golay: smooth residual noise, preserve step edges better than MA.
    # y_smooth = savgol_filter(y, sg_win, sg_poly, mode='interp')
    y_smooth = savgol_filter(y, sg_win, sg_poly, mode='interp')
    # y_smooth = y_med

    # --- 3) Percentile clipping to remove extreme outliers before clustering ---
    # Outliers can drag K-Means centroids and corrupt level estimation. Clip to
    # the 1st and 99th percentiles to limit their influence.
    p1, p99 = np.percentile(y_smooth, [1, 99])
    y_clip = np.clip(y_smooth, p1, p99)

    # --- 4) Two-cluster K-Means; use per-cluster medians as level estimates ---
    # We cluster the clipped smoothed signal into two groups (idle/active).
    # Instead of the centroid mean, we take the per-cluster *median* which is
    # more robust under asymmetrical distributions (sawtooth).
    km = KMeans(n_clusters=2, n_init='auto', random_state=0)
    km.fit(y_clip.reshape(-1, 1))
    labels = km.labels_
    c0 = np.mean(y_clip[labels == 0])
    c1 = np.mean(y_clip[labels == 1])
    mu_low, mu_high = (c0, c1) if c0 < c1 else (c1, c0)
    print(f"Estimated levels: mu_low={mu_low:.2f} W, mu_high={mu_high:.2f} W")

    # --- 5) Build robust settling bands using MAD + relative and absolute floors ---
    # The band half-width is the maximum of:
    #   (a) relative fraction of the level (band_frac_* * center),
    #   (b) absolute floor in watts (band_min_watts),
    #   (c) 1.4826 * MAD  (robust spread similar to sigma for Gaussian).
    def robust_band(vals, center, frac, min_w):
        mad = np.median(np.abs(vals - np.median(vals))) + 1e-9
        half = max(frac * center, min_w, 1.4826 * mad)
        return (center - half, center + half)

    # Split samples into "likely low" and "likely high" using the midpoint as a cut.
    low_vals  = y_clip[y_clip <= (mu_low + 0.5*(mu_high-mu_low))]
    high_vals = y_clip[y_clip >= (mu_low + 0.5*(mu_high-mu_low))]
    band_low  = robust_band(low_vals,  mu_low,  band_frac_low,  band_min_watts)
    band_high = robust_band(high_vals, mu_high, band_frac_high, band_min_watts)

    # --- 6) Hysteresis thresholds between the two levels ---
    # th_up defines when we "enter active" coming from idle.
    # th_dn defines when we "enter idle" coming from active.
    # Typically choose beta_dn < alpha_up to create hysteresis.
    th_up = mu_low + alpha_up*(mu_high - mu_low)
    th_dn = mu_low + beta_dn*(mu_high - mu_low)

    # --- 7) Binary state machine (0=idle, 1=active) driven by hysteresis ---
    # This avoids chattering caused by the sawtooth around the thresholds.
    state = np.zeros_like(y_smooth, dtype=int)
    cur = 0
    for i, val in enumerate(y_smooth):
        if cur == 0 and val >= th_up: cur = 1
        elif cur == 1 and val <= th_dn: cur = 0
        state[i] = cur

    # Convert time-based parameters to samples using the median timestep `dt`.
    dt = float(np.median(np.diff(x)))
    hold_n = max(1, int(round(hold_time / dt)))   # samples required to be continuously in-band
    min_gap_n = max(1, int(round(min_gap / dt)))  # skip-ahead after closing an event

    # Helper: starting from `idx_start`, find the earliest index where the next
    # `hold_n` samples all lie inside [lo, hi]. Return the last index of that
    # hold window (j-1), or None if such a window doesn't exist until the end.
    def settles(idx_start, lo, hi):
        i = idx_start
        while i < len(y_smooth):
            j = i + hold_n
            if j > len(y_smooth): return None
            seg = y_smooth[i:j]
            if np.all((seg >= lo) & (seg <= hi)):
                return j-1
            i += 1
        return None

    # Scan the state array and extract all rise (0->1) and fall (1->0) events.
    # For each event, compute t_start at the state flip and t_end at the first
    # index where the signal remains in-band for `hold_n` samples.
    def extract_events():
        events = []
        i = 1
        while i < len(state):
            # Rising edge: idle -> active
            if state[i-1]==0 and state[i]==1:
                t_start = x[i]
                j = settles(i, *band_high)
                if j is not None:
                    t_end = x[j]
                    events.append(dict(type='rise', t_start=t_start, t_end=t_end,
                                       duration=t_end - t_start))
                    i = j + min_gap_n
                    continue
            # Falling edge: active -> idle
            if state[i-1]==1 and state[i]==0:
                t_start = x[i]
                j = settles(i, *band_low)
                if j is not None:
                    t_end = x[j]
                    events.append(dict(type='fall', t_start=t_start, t_end=t_end,
                                       duration=t_end - t_start))
                    i = j + min_gap_n
                    continue
            i += 1
        return events
    print("Extracting events...")
    events = extract_events()
    print(f"Found {len(events)} events in first pass.")

    # --- 8) Optional second pass if nothing was found ---
    # If the bands were too tight and prevented settling, widen them slightly
    # and reduce the hold time; then try extraction again.
    if debug_second_pass and len(events) == 0:
        print("No events found; running second pass with relaxed bands and hold time...")
        band_frac_low2  = max(band_frac_low, 0.15)
        band_frac_high2 = max(band_frac_high, 0.12)
        band_low  = robust_band(low_vals,  mu_low,  band_frac_low2,  band_min_watts)
        band_high = robust_band(high_vals, mu_high, band_frac_high2, band_min_watts)
        hold_n = max(1, int(round(max(0.004, hold_time*0.5) / dt)))
        events = extract_events()
        print(f"Found {len(events)} events in second pass.")

    # ------------- Verification prints (safe, non-intrusive) -------------
    print(
        f"[levels] mu_low={mu_low:.3f} W, mu_high={mu_high:.3f} W | "
        f"th_up={th_up:.3f} W, th_dn={th_dn:.3f} W"
    )
    print(
        f"[bands]  low=({band_low[0]:.3f},{band_low[1]:.3f}) W, "
        f"high=({band_high[0]:.3f},{band_high[1]:.3f}) W | "
        f"dt={dt*1e3:.3f} ms, hold_n={hold_n}, min_gap_n={min_gap_n}"
    )
    if plot:
        # --- 9) Plot: raw + smoothed + thresholds + bands + events ---
        plt.figure(figsize=(12, 6))
        plt.plot(x, y, lw=0.7, alpha=0.35, label='Power (raw)')
        plt.plot(x, y_smooth, lw=1.2, label='Power (smoothed)')
        # Hysteresis thresholds:
        plt.axhline(th_up, color='tab:green', ls='--', lw=1, label='th_up (enter active)')
        plt.axhline(th_dn, color='tab:orange', ls='--', lw=1, label='th_dn (enter idle)')
        # Settling bands:
        plt.fill_between(x, band_high[0], band_high[1], color='tab:green', alpha=0.10, label='active band (settle)')
        plt.fill_between(x, band_low[0],  band_low[1],  color='tab:orange', alpha=0.10, label='idle band (settle)')

        # Event markers and shaded transition durations:
        for k, ev in enumerate(events):
            col = 'red' if ev['type']=='rise' else 'purple'
            plt.axvline(ev['t_start'], color=col, ls='--', lw=1,
                        label=('start rise' if ev['type']=='rise' and k==0 else
                            'start fall' if ev['type']=='fall' and k==0 else ""))
            plt.axvline(ev['t_end'], color=col, ls='-', lw=1,
                        label=('end rise' if ev['type']=='rise' and k==0 else
                            'end fall' if ev['type']=='fall' and k==0 else ""))
            plt.axvspan(ev['t_start'], ev['t_end'], color=col, alpha=0.12)

        plt.title('GPU Power: transitions and durations (robust levels + hysteresis + hold)')
        plt.xlabel('Time (s)')
        plt.ylabel('Power (W)')
        plt.grid(alpha=0.3)
        plt.tight_layout()
        if call_graph is not None:
            # --- 10) Overlay MPI/HIP call starts (if present) --------------------
            # We overlay vertical dotted lines at the *start time* of selected runtime calls.
            # This helps eyeball-corroborate whether state changes line up with kernel launches, etc.
            # Notes:
            # - The code expects `traces[TRACE][1]` to be a DataFrame with columns:
            #     ['Name', 'Start Time', ...]
            # - Only calls listed in `calls_to_mpi` or `calls_to_hip` are plotted.
            # - We further restrict to the visible interval [t_min, t_max].
            # - If your DataFrame uses different column names, adapt the filters accordingly.
            # - If there are many events, the legend might get dense; feel free to remove labels.
            # calls_to_mpi = ['MPI_Init', 'MPI_Finalize', 'MPI_Send', 'MPI_Recv', 'MPI_Isend', 'MPI_Irecv', 'MPI_Wait']
            calls_to_mpi = []
            calls_to_hip = ['hipLaunchKernel']
            call_graph = call_graph[(call_graph['Start Time'] >= t_min) & (call_graph['Start Time'] <= t_max)]
            call_graph = call_graph[call_graph['Name'].isin(calls_to_mpi + calls_to_hip)]
            print(f"Overlaying {len(call_graph)} MPI/HIP calls on the plot.")
            last_func = -1
            for index, row in call_graph.iterrows():
                if last_func >= 0 and row['Start Time'] - last_func < 0.050:
                    # Skip this launch to avoid overplotting
                    continue
                # Shade the region from the start to end time
                x = [row['Start Time'], row['End Time']]
                y = [0, max(y_smooth)*1.1]
                if row['Name'] in calls_to_mpi:
                    last_func = row['Start Time']
                    plt.fill_between(x, 0, y[1], color='cyan', alpha=0.025,
                                    label=row['Name'] if index == call_graph.index[0] else "")
                elif row['Name'] in calls_to_hip:
                    last_func = row['Start Time']
                    plt.fill_between(x, 0, y[1], color='cornflowerblue', alpha=0.025,
                                    label=row['Name'] if index == call_graph.index[0] else "")

        # Final legend call (after overlay lines so labels are included if any)
        plt.legend(ncol=2)
        # Cut the plot to the visible range
        plt.xlim(vt_min, vt_max)
        plt.ylim(0, None)
        plt.show()
    elif call_graph is None and plot:
        print("No call graph provided; skipping overlay of MPI/HIP calls.")
    if summary:
        # Console summary of detected transitions (start, end, duration in ms)
        for ev in events:
            print(f"{ev['type']:>4}  start={ev['t_start']:.6f}s  end={ev['t_end']:.6f}s  "
                f"duration={ev['duration']*1e3:.3f} ms")
        print('Average rising duration: {:.3f} ms'.format(
            sum(ev['duration']*1e3 for ev in events if ev['type'] == 'rise') /
            max(sum(1 for ev in events if ev['type'] == 'rise'), 1)))
        print('Average falling duration: {:.3f} ms'.format(
            sum(ev['duration']*1e3 for ev in events if ev['type'] == 'fall') /
            max(sum(1 for ev in events if ev['type'] == 'fall'), 1)))
    return events

In [ ]:
def calculate_average_rising_and_falling_durations(events: List[Dict[str, Any]]) -> Tuple[float, float]:
    rising_durations = [ev['duration'] for ev in events if ev['type'] == 'rise']
    falling_durations = [ev['duration'] for ev in events if ev['type'] == 'fall']
    avg_rising = sum(rising_durations) / len(rising_durations) if rising_durations else 0.0
    avg_falling = sum(falling_durations) / len(falling_durations) if falling_durations else 0.0
    return avg_rising, avg_falling

# events = calculate_rising_and_falling_events(single_gcd_runs, alpha_up=0.1, beta_dn=0.7, vt_min=2, vt_max=13, hold_time=0.003, t_min=2, t_max=13, plot=False, summary=False)
# avg_rising, avg_falling = calculate_average_rising_and_falling_durations(events)
# print(f"Average Rising Duration: {avg_rising*1e3:.3f} ms")
# print(f"Average Falling Duration: {avg_falling*1e3:.3f} ms")

In [ ]:
def calculate_average_rising_and_falling_durations_across_ensembles(traces: List[Ensemble], node_name: str, metric_name: str) -> Tuple[float, float]:
    rising_durations = []
    falling_durations = []
    for trace in traces:
        for run in trace.runs:
            call_graph = run.nodes[0].ranks[0].call_graph
            events = calculate_rising_and_falling_events(run.get_metric_samples(node_name, metric_name), key=metric_name, call_graph=call_graph, vt_min=2, vt_max=13, hold_time=0.003, t_min=2, t_max=15, plot=True, summary=True)
            avg_rising, avg_falling = calculate_average_rising_and_falling_durations(events)
            rising_durations.append(avg_rising)
            falling_durations.append(avg_falling)
    overall_avg_rising = sum(rising_durations) / len(rising_durations) if rising_durations else 0.0
    overall_avg_falling = sum(falling_durations) / len(falling_durations) if falling_durations else 0.0
    return overall_avg_rising, overall_avg_falling

calculate_rising_and_falling_events(single_gcd_runs[(1000, 1000)].runs[0].get_metric_samples('node0', 'A2rocm_smi:::energy_count:device=0'), vt_min=0, vt_max=15, beta_dn=0.85, alpha_up=0.15, key='A2rocm_smi:::energy_count:device=0', hold_time=0.003, plot=True, summary=True)
calculate_rising_and_falling_events(single_gcd_runs[(1000, 1000)].runs[0].get_metric_samples('node0', 'A2rocm_smi:::energy_count:device=0'), vt_min=2, vt_max=4, beta_dn=0.85, alpha_up=0.15, key='A2rocm_smi:::energy_count:device=0', hold_time=0.003, plot=True, summary=True)
calculate_rising_and_falling_events(single_gcd_runs[(1000, 1000)].runs[0].get_metric_samples('node0', 'A2rocm_smi:::energy_count:device=0'), vt_min=3.25, vt_max=3.5, beta_dn=0.85, alpha_up=0.15, key='A2rocm_smi:::energy_count:device=0', hold_time=0.003, plot=True, summary=True)

In [ ]:
calculate_average_rising_and_falling_durations_across_ensembles([single_gcd_runs[(1000, 1000)]], 'node0', 'A2rocm_smi:::energy_count:device=2')

In [ ]:

from typing import List, Literal, Optional
import numpy as np

Label = Literal["rising", "falling", "peak", "trough"]

def label_by_fixed_bands_xy(
    x: Optional[List[float]],
    y: List[float],
    band_frac: float = 1/4,     # each plateau band takes this fraction of (max-min)
    min_trough_run: int = 1,    # squash trough runs shorter than this
    min_peak_run: int = 1       # squash peak runs shorter than this
) -> List[Label]:
    """
    Amplitude-only labels using fixed bands from [min(y), max(y)].

    trough: y <= ymin + band_frac*(ymax - ymin)
    peak:   y >= ymax - band_frac*(ymax - ymin)
    middle: everything else -> 'rising'/'falling' based on surrounding plateaus.
    """
    n = len(y)
    if x is not None and len(x) != n:
        raise ValueError("x and y must have the same length when x is provided.")
    if n == 0:
        return []
    if n == 1:
        return ["peak"]

    # Preserve order (sort by time if provided)
    order = np.argsort(x) if x is not None else np.arange(n)
    inv = np.empty_like(order); inv[order] = np.arange(n)
    y_arr = np.asarray(y, float)[order]
    m = len(y_arr)

    ymin = float(np.min(y_arr))
    ymax = float(np.max(y_arr))
    yr = ymax - ymin
    if yr <= 0.0:
        return ["peak"] * n  # flat signal ⇒ arbitrary single plateau

    # Ensure bands don't overlap (cap band_frac at < 0.5)
    bf = float(np.clip(band_frac, 0.0, 0.49))
    thr_low  = ymin + bf * yr
    thr_high = ymax - bf * yr

    is_trough = y_arr <= thr_low
    is_peak   = y_arr >= thr_high
    # If any point is somehow both (numerical tie), prefer the closer side
    both = is_trough & is_peak
    if both.any():
        closer_peak = (y_arr[both] - thr_high) <= (thr_low - y_arr[both])
        is_peak[both] = closer_peak
        is_trough[both] = ~closer_peak

    # Simple run squashing
    def squash(mask: np.ndarray, min_run: int) -> np.ndarray:
        if min_run <= 1:
            return mask
        out = mask.copy()
        i = 0
        while i < len(out):
            if out[i]:
                j = i
                while j < len(out) and out[j]:
                    j += 1
                if (j - i) < min_run:
                    out[i:j] = False
                i = j
            else:
                i += 1
        return out

    is_trough = squash(is_trough, min_trough_run)
    is_peak   = squash(is_peak,   min_peak_run)

    # Fill middle band using the sandwich rule
    labels = np.array(["rising"] * m, dtype=object)
    labels[is_trough] = "trough"
    labels[is_peak]   = "peak"

    left = np.full(m, "", dtype=object)
    right = np.full(m, "", dtype=object)

    last = ""
    for i in range(m):
        if labels[i] in ("peak", "trough"):
            last = labels[i]
        left[i] = last

    last = ""
    for i in range(m-1, -1, -1):
        if labels[i] in ("peak", "trough"):
            last = labels[i]
        right[i] = last

    for i in range(m):
        if labels[i] in ("peak", "trough"):
            continue
        L, R = left[i], right[i]
        if L == "trough" and R == "peak":
            labels[i] = "rising"
        elif L == "peak" and R == "trough":
            labels[i] = "falling"
        elif L == "peak" and R == "peak":
            labels[i] = "peak"
        elif L == "trough" and R == "trough":
            labels[i] = "trough"
        elif L == "peak" and R == "":
            labels[i] = "falling"
        elif L == "trough" and R == "":
            labels[i] = "rising"
        else:
            # no context at one edge: bias by proximity to thresholds
            labels[i] = "rising" if abs(y_arr[i] - thr_high) < abs(y_arr[i] - thr_low) else "falling"

    return labels[inv].tolist()

def calculate_periods_from_run(run: Run, debug: bool = False, metric="A2rocm_smi:::energy_count:device=0", differentiate=True) -> int:
    table = run.get_metric_samples('node0', metric)
    table = table[(table['Time'] >= 1)]
    signal_x = np.array(table['Time'])
    signal_f = sampled_to_continuous(table)
    if differentiate:
        signal_fprime = numerical_derivative(signal_f, h=0.0005)
        signal_y = np.array(signal_fprime(signal_x))
    else:
        signal_y = np.array(signal_f(signal_x))

    labels = label_by_fixed_bands_xy(signal_x, signal_y, band_frac=1/4)
    
    # Count the number of periods based on peak/trough labels
    count = 0
    last_label = ""
    for label in labels:
        if label in ("peak", "trough") and label != last_label:
            print(f"Detected {label} region.")
            count += 1
        last_label = label
    if debug:
        plt.figure(figsize=(32, 4))
        plt.plot(signal_x, signal_y, label='Signal Derivative', color='blue', alpha=0.7)
        plt.title('Signal Derivative with Peak/Trough Annotations')
        plt.xlabel('Time (s)')
        plt.ylabel('Derivative Value')
        
        # Filter for the peaks
        peak_indices = [i for i, label in enumerate(labels) if label == "peak"]
        trough_indices = [i for i, label in enumerate(labels) if label == "trough"]
        plt.scatter(signal_x[peak_indices], signal_y[peak_indices], color='orange', label='Peak', s=50, marker='^')
        plt.scatter(signal_x[trough_indices], signal_y[trough_indices], color='purple', label='Trough', s=50, marker='v')
        
        # for i, label in enumerate(labels):
        #     if label == "peak":
        #         plt.scatter(signal_x[i], signal_y[i], color='orange', label='Peak' if i == 0 else "", s=50, marker='^')
        #     elif label == "trough":
        #         plt.scatter(signal_x[i], signal_y[i], color='purple', label='Trough' if i == 0 else "", s=50, marker='v')
        plt.legend()
        plt.grid(alpha=0.3)
        plt.show()
    return count // 2  # each period has a peak and a trough

# # # Example usage of label_trend_regions
# table = single_gcd_runs[(1000, 1000)].runs[0].get_metric_samples('node0', 'A2rocm_smi:::energy_count:device=0')
# table = table[(table['Time'] >= 1)]
# signal_x = np.array(table['Time'])
# signal_f = sampled_to_continuous(table)
# signal_fprime = numerical_derivative(signal_f, h=0.0005)
# signal_y = np.array(signal_fprime(signal_x))


# labels = label_by_fixed_bands_xy(signal_x, signal_y)
# for i, label in enumerate(labels):
#     print(f"Sample {i}: {label}")
    
# # Graph it but color the regions based on labels
# plt.figure(figsize=(12, 4))
# unique_labels = ["rising", "falling", "peak", "trough"]
# colors = {
#     "rising": "green",
#     "falling": "red",
#     "peak": "orange",
#     "trough": "purple"
# }
# for label in unique_labels:
#     indices = np.array([i for i, l in enumerate(labels) if l == label])
#     plt.scatter(signal_x[indices], signal_y[indices], label=label, color=colors[label], s=10)
# plt.title('Labeled Trend Regions in Signal')
# plt.xlabel('Time (s)')
# plt.ylabel('Signal Value')
# plt.grid(alpha=0.3)
# plt.legend()
# plt.show()


# Count the number of transitions. Dont count consecutive identical labels as transitions.
def count_transitions(labels: List[Label], transition_type: str) -> int:
    count = 0
    last_label = ""
    for label in labels:
        if label == transition_type and last_label != transition_type:
            count += 1
        last_label = label
    return count
# num_rising = count_transitions(labels, "rising")
# num_falling = count_transitions(labels, "falling")
# print(f"Number of rising transitions: {num_rising}")
# print(f"Number of falling transitions: {num_falling}")


def count_periods(labels: List[Label]) -> int:
    # Count:
    # the number of contiguous peak regions
    # the number of contiguous trough regions
    count = 0
    last_label = ""
    for label in labels:
        if label in ("peak", "trough") and label != last_label:
            print(f"Detected {label} region.")
            count += 1
        last_label = label
    return count

# num_periods = count_periods(labels)
# print(f"Number of periods (peak/trough): {calculate_periods_from_run(single_gcd_runs[(5, 1000)].runs[0])}")
# print(f"Correct number of periods: {calculate_steps_from_periods(5, 1000)}")
calculate_periods_from_run(single_gcd_runs[(2, 20)].runs[0], debug=True)

# Create a heatmap of the percent error between detected periods and expected periods
single_gcd_heatmap = pd.DataFrame(index=ACTIVE_PERIODS_MS, columns=IDLE_PERIODS_MS)
for time_active in ACTIVE_PERIODS_MS:
    for time_idle in IDLE_PERIODS_MS:
        expected_periods = calculate_steps_from_periods(time_active, time_idle)
        detected_periods = calculate_periods_from_run(single_gcd_runs[(time_active, time_idle)].runs[0])
        percent_error = abs(detected_periods - expected_periods) / expected_periods * 100 if expected_periods > 0 else 0
        print(f"Active: {time_active} ms, Idle: {time_idle} ms | Expected: {expected_periods}, Detected: {detected_periods}, Percent Error: {percent_error:.2f}%")
        single_gcd_heatmap.at[time_active, time_idle] = percent_error
# Now graph the heatmap
plt.figure(figsize=(10, 8))
# Sort the index and columns
single_gcd_heatmap = single_gcd_heatmap.sort_index(ascending=False).sort_index(axis=1)
sns.heatmap(single_gcd_heatmap.astype(float), annot=True, fmt=".1f", cmap="RdYlGn_r", cbar_kws={'label': 'Percent Error (%)'})
plt.title("Heatmap of Percent Error in Period Detection (ROCm)")
plt.xlabel("Idle Period (ms)")
plt.ylabel("Active Period (ms)")
plt.show()


# Create a heatmap of the percent error between detected periods and expected periods
single_gcd_heatmap = pd.DataFrame(index=ACTIVE_PERIODS_MS, columns=IDLE_PERIODS_MS)
for time_active in ACTIVE_PERIODS_MS:
    for time_idle in IDLE_PERIODS_MS:
        expected_periods = calculate_steps_from_periods(time_active, time_idle)
        detected_periods = calculate_periods_from_run(single_gcd_runs[(time_active, time_idle)].runs[0], metric='A2coretemp:::craypm:accel0_power', differentiate=False)
        percent_error = abs(detected_periods - expected_periods) / expected_periods * 100 if expected_periods > 0 else 0
        print(f"Active: {time_active} ms, Idle: {time_idle} ms | Expected: {expected_periods}, Detected: {detected_periods}, Percent Error: {percent_error:.2f}%")
        single_gcd_heatmap.at[time_active, time_idle] = percent_error

# Now graph the heatmap
plt.figure(figsize=(10, 8))
# Sort the index and columns
single_gcd_heatmap = single_gcd_heatmap.sort_index(ascending=False).sort_index(axis=1)
sns.heatmap(single_gcd_heatmap.astype(float), annot=True, fmt=".1f", cmap="RdYlGn_r", cbar_kws={'label': 'Percent Error (%)'})
plt.title("Heatmap of Percent Error in Period Detection (CrayPM)")
plt.xlabel("Idle Period (ms)")
plt.ylabel("Active Period (ms)")
plt.show()


In [ ]:
# Get the end time of the notebook
NOTEBOOK_END_TIME = time.time()
print(f"Notebook end time: {NOTEBOOK_END_TIME}")
print(f"Notebook end at {NOTEBOOK_END_TIME} ({datetime.datetime.fromtimestamp(NOTEBOOK_END_TIME)})")

NOTEBOOK_DURATION = NOTEBOOK_END_TIME - NOTEBOOK_START_TIME
print(f"Notebook execution time: {NOTEBOOK_DURATION:.2f} seconds")